# Phase 2 Model Evaluation, Error Analysis & Visualization
**Project:** Air Quality Forecasting via Gradient Boosting and LSTM Models  
**Role:** Model Evaluation, Error Analysis and Visualization  
**Author:** Mihika Mathur  
**Scope:** Multi-horizon LSTM evaluation across all 12 Beijing monitoring stations and 5 forecast horizons (1h, 3h, 6h, 12h, 24h), with rigorous comparative benchmarking against Phase 1 models.  

---
## Notebook Objectives
1. **Multi-Horizon LSTM Evaluation**: Quantitative assessment of test-set predictions across 12 stations and 5 forecast horizons (t+1h, t+3h, t+6h, t+12h, t+24h).
2. **Phase 1 vs. Phase 2 Benchmarking**: Direct head-to-head comparison against Phase 1 models (Naive Persistence, Linear Regression, Default Gradient Boosting, Tuned Gradient Boosting).
3. **Macro-Averaging and Observation-Pooled Metrics**: Rigorous aggregation demonstrating that pooled RMSE mathematically derives from pooled squared errors.
4. **Error Analysis and Stratifications**: Diagnostics across pollution concentration tiers (WHO/AQI bins), diurnal cycles (0-23h), monthly patterns, and extreme haze episodes.
5. **Publication Visualizations**: Generation of all 8 core evaluation figures.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set repository root in sys.path
repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.phase2.evaluate import evaluate_all_stations, save_evaluation_artifacts
from src.phase2.visualize import generate_all_evaluation_figures

print(f"Project repository root: {repo_root}")


## 1. Execute Multi-Station Multi-Horizon Evaluation Suite
We evaluate all 12 stations across horizons [1, 3, 6, 12, 24] hours using the champion LSTM checkpoint (=72 lookback, 64 units, 1 recurrent layer, MAE loss).

In [2]:
suite = evaluate_all_stations(window_size=72, partition="test")
saved_files = save_evaluation_artifacts(suite)
print("Evaluation complete across", len(suite.stations), "monitoring stations.")
print("Total test observations evaluated per horizon:", suite.macro_summary_df["total_observations"].iloc[0])


## 2. Macro-Average Results Across Stations
The table below summarizes macro-averaged metrics (arithmetic mean across the 12 stations) for each horizon.

In [3]:
cols_macro = [
    "horizon", "lstm_macro_mae", "pers_macro_mae", "macro_mae_improvement_pct",
    "lstm_macro_rmse", "pers_macro_rmse", "macro_rmse_improvement_pct",
    "lstm_macro_r2", "pers_macro_r2", "macro_r2_delta", "lstm_macro_mbe"
]
suite.macro_summary_df[cols_macro]


## 3. Observation-Pooled Results
Pooled metrics are calculated over all 58,815 test observations across Beijing.
**Note:** Pooled RMSE is strictly calculated as $\sqrt{rac{1}{N} \sum (y - \hat{y})^2}$, NOT by averaging station RMSEs.

In [4]:
cols_pooled = [
    "horizon", "pooled_mae_lstm", "pooled_mae_pers", "pooled_mae_improvement_pct",
    "pooled_rmse_lstm", "pooled_rmse_pers", "pooled_rmse_improvement_pct",
    "pooled_r2_lstm", "pooled_r2_pers", "pooled_r2_delta", "pooled_mbe_lstm"
]
suite.pooled_summary_df[cols_pooled]


## 4. Phase 1 vs. Phase 2 Comparative Benchmark (1-Hour Horizon)
We align Phase 2 1-hour LSTM performance directly against Phase 1 models on compatible test periods.

In [5]:
p1_summary = suite.phase1_comparison_1h_df.groupby("model")[
    ["test_mae", "test_rmse", "test_r2", "test_mbe", "mae_improvement_pct"]
].mean().reset_index().sort_values(by="test_mae")
p1_summary


## 5. Stratified Error Diagnostics
### 5.1 Error by Concentration Tier (WHO / AQI Tiers)
Assessing whether the model systematically underpredicts acute episodic peaks.

In [6]:
tier_1h = suite.tier_error_df[suite.tier_error_df["horizon"] == "1h"]
tier_1h[["tier", "concentration_range", "mae", "rmse", "mbe", "n_samples", "pct_of_test_data"]]


### 5.2 Diurnal Error Profile (Hour of Day 0-23)
Evaluating diurnal variation in MAE and MBE.

In [7]:
diurnal_1h = suite.diurnal_error_df[suite.diurnal_error_df["horizon"] == "1h"]
plt.figure(figsize=(10, 4))
plt.plot(diurnal_1h["hour_of_day"], diurnal_1h["mae"], marker="o", label="1h MAE")
plt.plot(diurnal_1h["hour_of_day"], diurnal_1h["mbe"], marker="s", label="1h MBE")
plt.axhline(0, color="black", linestyle="--", alpha=0.5)
plt.xlabel("Hour of Day (0-23)")
plt.ylabel("Metric (µg/m³)")
plt.title("Diurnal Error Progression")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.show()


## 6. Publication Visualization Suite
Generating and displaying the 8 publication-grade diagnostic figures.

In [8]:
figs = generate_all_evaluation_figures(suite)
print("Generated figures:")
for k, path in figs.items():
    print(f"  - {k}: {path}")
